In [1]:
# Building the chaperone/protease census (Day 1 deliverable).
#
# The census is what Claim 1 is counted against, so it was frozen before any
# comparison with ATFS-1 target data and has not been touched since. This notebook
# reconstructs it from the annotation file and asserts that it still reproduces the
# frozen CSV exactly - it does not rebuild or overwrite it. If a future WormBase
# release changes a domain call, this notebook fails loudly rather than silently
# shifting the number the paper reports.
#
# WormBase's gene-class pages are blocked to scripted access, so membership is
# decided from Pfam protein-domain annotations instead. That is the better basis
# anyway: it does not depend on a gene happening to be named hsp-*, and it catches
# renamed genes under their current identity (daf-21 is hsp-90, same WBGene ID).
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

GFF = "data/raw/c_elegans.PRJNA13758.WS285.protein_annotation.gff3.gz"
FROZEN = "data/chaperone_protease_census.csv"

# The inclusion rule, written down before the count was known. Eight families
# qualify on a single domain; Lon is the exception and needs both of its domains
# (see the next cell for why that matters). Domain names are the real Pfam family
# names pulled from the annotation file itself, checked directly rather than
# assumed - two entries here were wrong in an earlier version of this cell
# (PF01434 as "FtsH_AAA", PF00574 as "ClpP") because they were written from memory
# as human-readable mnemonics instead of read off the file. The real names are
# Peptidase_M41 and CLP_protease. The exact-match check below did not catch this
# the first time because it only compared gene membership and role, not domain-name
# text; it now checks the text too.
SINGLE_DOMAIN = {
    "PF00012": "HSP70",       "PF00226": "DnaJ",
    "PF00183": "HSP90",       "PF00011": "HSP20",
    "PF00118": "Cpn60_TCP1",  "PF01920": "Prefoldin",
    "PF00574": "CLP_protease", "PF01434": "Peptidase_M41",
}
LON_DOMAINS = {"PF05362": "Lon_C", "PF02190": "LON_substr_bdg"}
PROTEASE_DOMAINS = {"PF00574", "PF01434", "PF05362", "PF02190"}
ALL_DOMAINS = set(SINGLE_DOMAIN) | set(LON_DOMAINS)

print(f"Inclusion rule: {len(SINGLE_DOMAIN)} single-domain families, "
      f"plus Lon on {' + '.join(LON_DOMAINS)}")

Inclusion rule: 8 single-domain families, plus Lon on PF05362 + PF02190


In [2]:
# Read the annotation once, keeping three things per protein isoform: which gene it
# belongs to, which of the target domains it carries, and whether it has an ER
# signal peptide.
#
# Everything here is per-isoform rather than per-gene, and that turns out to matter
# in both directions - see the ER screen below.
protein_gene, protein_domains, has_signal_peptide = {}, {}, set()

with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue
        protein, source, feature, attrs = fields[0], fields[1], fields[2], fields[8]

        if source == "WormBase" and feature == "CDS":
            a = dict(kv.split("=", 1) for kv in attrs.split(";") if "=" in kv)
            if "wormbase_geneid" in a:
                protein_gene[protein] = (a["wormbase_geneid"],
                                         a.get("wormbase_genename", ""))
        elif source == "Pfam" and feature == "motif":
            for pfam in ALL_DOMAINS:
                if pfam in attrs:
                    protein_domains.setdefault(protein, set()).add(pfam)
        elif source == "SignalP" and feature == "signal_peptide":
            has_signal_peptide.add(protein)

print(f"Protein isoforms with a gene mapping: {len(protein_gene):,}")
print(f"Isoforms carrying at least one target domain: {len(protein_domains):,}")
print(f"Isoforms with an ER signal peptide (all genes): {len(has_signal_peptide):,}")

Protein isoforms with a gene mapping: 28,558
Isoforms carrying at least one target domain: 128
Isoforms with an ER signal peptide (all genes): 5,156


In [3]:
# Qualification, and the ER screen.
#
# Lon needs both domains. lonp-1 and lonp-2 carry Lon_C and LON_substr_bdg together;
# five other genes carry a lone Lon_C hit and are not Lon proteases. Accepting either
# domain on its own pulls all five in.
#
# The ER screen asks whether a signal peptide sits on the *same protein* as the
# qualifying domain, not merely somewhere in the gene. Both halves of that matter:
#
#   hsp-3, hsp-4, enpl-1 - the signal peptide and the chaperone domain are on the
#   same isoform. These are real ER chaperones and belong out of scope. Their short
#   isoforms lack the peptide only because they start downstream of it, so a rule
#   that kept a gene whenever any isoform looked clean would wrongly retain them.
#
#   F11F1.1 - the signal peptide is on isoform c, which carries DUF148 and no
#   chaperone domain at all; the HSP70 domain is on isoforms a and b, neither of
#   which has a peptide. So a rule that dropped a gene whenever any isoform carried
#   a peptide would wrongly discard it.
qualifying, er_targeted = {}, set()
for protein, domains in protein_domains.items():
    if protein not in protein_gene:
        continue
    matched = domains & set(SINGLE_DOMAIN)
    if set(LON_DOMAINS) <= domains:
        matched |= set(LON_DOMAINS)
    if not matched:
        continue

    gene_id, gene_name = protein_gene[protein]
    if protein in has_signal_peptide:
        er_targeted.add(gene_id)
    entry = qualifying.setdefault(gene_id, {"name": gene_name, "domains": set()})
    entry["domains"] |= matched

print(f"Genes carrying a qualifying domain: {len(qualifying)}")
print(f"Excluded as ER-targeted: {len(er_targeted)}")
print("  " + ", ".join(sorted(qualifying[g]["name"] or g for g in er_targeted)))

census = {g: e for g, e in qualifying.items() if g not in er_targeted}
print(f"Remaining after the ER screen: {len(census)}")

Genes carrying a qualifying domain: 85
Excluded as ER-targeted: 13
  F54F2.9, T14G8.3, T24H7.2, dnj-2, dnj-20, dnj-27, dnj-28, dnj-7, dnj-8, enpl-1, hsp-3, hsp-4, stc-1
Remaining after the ER screen: 72


In [4]:
# Two manual adjustments, both recorded in gate_decisions.md and both checked
# against the annotation rather than assumed.
#
# Removals: genes where the matched domain is a minor accessory feature rather than
# the protein's function. ppk-3 is a >1,400-residue PIKfyve lipid kinase carrying a
# weak chaperonin hit; rme-8 is an endosomal trafficking protein with a small
# accessory DnaJ domain, the same category as the PEX19-type exclusions. lido-17 was
# the third gene on this list, but it only ever qualified through a lone Lon_C hit,
# so requiring both Lon domains already removes it - it is kept in the list here so
# the rule stays explicit, and the assertion below confirms it never had to fire.
MANUAL_EXCLUSIONS = {"ppk-3", "lido-17", "rme-8"}
fired = sorted(e["name"] for e in census.values() if e["name"] in MANUAL_EXCLUSIONS)
census = {g: e for g, e in census.items() if e["name"] not in MANUAL_EXCLUSIONS}
print(f"Manual exclusions applied: {fired}")
print(f"  (lido-17 not listed above = already removed by the two-domain Lon rule)")

# Additions: prefoldin is a six-subunit complex and only four subunits match PF01920
# in this release. pfd-3 and pfd-5 carry no Pfam call of any kind here - they are
# short (~150-185 residues) and fall below this scan's threshold - but their identity
# as canonical subunits is not in question, unlike the borderline cases above. The
# frozen file records this reasoning directly in its own pfam_families text rather
# than just naming the domain, which the exact-match check now requires this
# rebuild to reproduce word for word, not only in substance.
name_to_gene = {}
for protein, (gene_id, gene_name) in protein_gene.items():
    if gene_name:
        name_to_gene.setdefault(gene_name, gene_id)

PREFOLDIN_MANUAL_NOTE = "Prefoldin (manual add - no Pfam hit in this release; canonical complex member)"
for subunit in ("pfd-3", "pfd-5"):
    gene_id = name_to_gene.get(subunit)
    if gene_id is None:
        raise RuntimeError(f"{subunit} is absent from the annotation entirely.")
    if gene_id in census:
        raise RuntimeError(f"{subunit} now matches automatically - the manual "
                           "addition is stale and should be removed.")
    census[gene_id] = {"name": subunit, "domains": {"PF01920"},
                       "label_override": PREFOLDIN_MANUAL_NOTE}

matched_subunits = sorted(e["name"] for e in census.values()
                          if e["name"].startswith("pfd-"))
print(f"Prefoldin subunits in the census: {matched_subunits}")
print(f"Census size: {len(census)}")

Manual exclusions applied: ['ppk-3', 'rme-8']
  (lido-17 not listed above = already removed by the two-domain Lon rule)
Prefoldin subunits in the census: ['pfd-1', 'pfd-2', 'pfd-3', 'pfd-4', 'pfd-5', 'pfd-6']
Census size: 72


In [5]:
# Does this still reproduce the frozen file? The census was frozen on 2026-08-12 and
# every downstream count in the paper is built on it, so this is the check that
# matters: not "does the rule produce something reasonable" but "does it produce
# exactly what the paper reports". Checked three ways, not just gene count: exact
# gene membership, exact role assignment, and exact domain-name text - the first
# version of this cell only checked the first two, which is how the FtsH_AAA/
# Peptidase_M41 and ClpP/CLP_protease naming mismatches above went unnoticed until
# reconciled by hand.
def role_of(domains):
    return "protease" if domains & PROTEASE_DOMAINS else "chaperone"

rebuilt = pd.DataFrame([
    {"gene_id": gene_id, "public_name": entry["name"],
     "role": role_of(entry["domains"]),
     "pfam_families": entry.get("label_override") or ",".join(sorted(
         {**SINGLE_DOMAIN, **LON_DOMAINS}[d] for d in entry["domains"]))}
    for gene_id, entry in census.items()
]).sort_values("gene_id").reset_index(drop=True)

frozen = pd.read_csv(FROZEN)
rebuilt_ids, frozen_ids = set(rebuilt["gene_id"]), set(frozen["gene_id"])

print(f"Rebuilt: {len(rebuilt_ids)} genes | Frozen: {len(frozen_ids)} genes")
missing, extra = frozen_ids - rebuilt_ids, rebuilt_ids - frozen_ids
if missing or extra:
    raise RuntimeError(
        f"Census no longer reproduces. In the frozen file but not rebuilt: "
        f"{sorted(missing)}. Rebuilt but not in the frozen file: {sorted(extra)}. "
        "Do not update the CSV to match - investigate why the rule moved.")

frozen_roles = frozen.set_index("gene_id")["role"].to_dict()
role_disagreements = [
    (r.gene_id, r.public_name, r.role, frozen_roles[r.gene_id])
    for r in rebuilt.itertuples() if frozen_roles[r.gene_id] != r.role
]
if role_disagreements:
    raise RuntimeError(f"Role assignments disagree: {role_disagreements}")

# Domain-name text, compared exactly (manual-add entries included), so the check
# cannot pass on gene count and role alone while the descriptive text drifts.
frozen_domains = frozen.set_index("gene_id")["pfam_families"].to_dict()
domain_disagreements = [
    (r.gene_id, r.public_name, r.pfam_families, frozen_domains[r.gene_id])
    for r in rebuilt.itertuples()
    if set(r.pfam_families.split(",")) != set(frozen_domains[r.gene_id].split(","))
]
if domain_disagreements:
    raise RuntimeError(f"Domain-name text disagrees: {domain_disagreements}")

counts = rebuilt["role"].value_counts()
print(f"Gene membership, role assignments, AND domain-name text all reproduce exactly: "
      f"{counts.get('chaperone', 0)} chaperones, {counts.get('protease', 0)} proteases.")

# The genes the roadmap named as must-be-present, checked individually rather than
# trusting the total. daf-21 is hsp-90's former name and the same WBGene ID.
for expected in ["hsp-6", "hsp-60", "dnj-10", "ymel-1", "spg-7", "clpp-1", "cct-1", "hsp-90"]:
    if expected not in set(rebuilt["public_name"]):
        raise RuntimeError(f"{expected} should be in the census and is not.")
print("Named sanity-check genes all present: hsp-6, hsp-60, dnj-10, ymel-1, "
      "spg-7, clpp-1, cct-1, hsp-90.")

# The roadmap's three borderline genes, confirmed absent on domain grounds. The
# permissive count in the paper (72 + 3) rests on a functional argument, not a
# domain one, and this is where that distinction is checked.
for borderline in ["prx-19", "cbp-3", "tspo-1"]:
    if borderline in set(rebuilt["public_name"]):
        raise RuntimeError(f"{borderline} is borderline and should not qualify on domains.")
print("Borderline genes correctly absent on domain evidence: prx-19, cbp-3, tspo-1.")

Rebuilt: 72 genes | Frozen: 72 genes
Gene membership, role assignments, AND domain-name text all reproduce exactly: 65 chaperones, 7 proteases.
Named sanity-check genes all present: hsp-6, hsp-60, dnj-10, ymel-1, spg-7, clpp-1, cct-1, hsp-90.
Borderline genes correctly absent on domain evidence: prx-19, cbp-3, tspo-1.
